# Build a stable softmax

Turn scores into probabilities without overflowing the exponential.

**Your exported case** · Two rows · 37 columns

**Predict for the starter case.** With 37 columns and 32 threads per row, which columns does Lane 0 write? What should each stable probability row sum to?

Run **Set up this notebook** once per kernel session. Then run **Your exported program**, or go straight to a numbered experiment. Each program cell contains its own kernel, inputs and launch. Shift+Enter runs that cell and checks that run’s returned array. The first line names the lesson, case and task. Leave it in place when editing Python below it.

Each numbered task opens with its own starting code, including any required earlier edits. If the task calls for a change, running this code checks the starting result. Make the edit and rerun the cell to check the changed program. To switch cases, run the cell under that case’s heading. To keep two versions, duplicate the cell before editing. Output belongs to the last execution of that cell. Rerun after each edit.

## Set up this notebook

Use a Python 3 IPython kernel in Jupyter or Colab. Reading the file and inspecting the references work offline. Executing CUDA programs needs an NVIDIA GPU and a working native backend. In Colab, choose **Runtime → Change runtime type → GPU** if one is available. For local Linux or Jetson, select the kernel for your CUDA environment.

Run the setup cell below to see the Python, architecture, compiler, driver, runtime and device it detects. For a fresh compatible CUDA 12 environment, review the package plan first in a separate cell:

```python
%pip install --dry-run "numba-cuda-mlir[cu12]==0.6.0" "numpy>=1.26,<2.5"
```

If the plan fits your environment, repeat without `--dry-run`, restart the kernel and rerun setup. Installation needs package access or an offline wheelhouse. On Jetson, record JetPack and check the ARM64 wheels against your Python and system CUDA versions before installing. A compatible wheel alone does not establish device support.

To recover a starting program, run `print(lab_start("CASE", "TASK"))` in a new cell, using the two IDs after the lesson name on its first line. Copy the printed Python into a duplicate of the program cell. Your original edits stay in place.

In [ ]:
# Register the offline experiment runner and check this environment.
"""Native lesson support embedded in each exported IPython notebook."""
import contextlib
import hashlib
import io
import json
import linecache
import platform
import re
import sys
from importlib import metadata
from pathlib import Path


def lab_preflight():
    """Report the running environment. No install, kernel launch or backend substitution."""
    print(f"Python {platform.python_version()} · {platform.platform()} · {sys.executable}")
    jetpack = Path("/etc/nv_tegra_release")
    if jetpack.is_file():
        print("Jetson Linux:", jetpack.read_text().splitlines()[0])
    versions = {}
    for package in ("numba-cuda-mlir", "numpy", "cuda-bindings", "cuda-toolkit", "nvidia-cuda-nvcc-cu12"):
        try:
            versions[package] = metadata.version(package)
        except metadata.PackageNotFoundError:
            versions[package] = "not installed"
        print(f"{package}: {versions[package]}")
    try:
        from cuda.bindings import runtime
        status, runtime_version = runtime.cudaRuntimeGetVersion()
        print(f"CUDA runtime API: {runtime_version} ({status})")
        if int(status) or not 12000 <= runtime_version < 13000:
            raise RuntimeError("The pinned cu12 environment needs a working CUDA 12 runtime.")
        from cuda.bindings import driver
        status, driver_version = driver.cuDriverGetVersion()
        print(f"Driver CUDA API: {driver_version} ({status})")
        if int(status) or driver_version < 12000:
            raise RuntimeError("The pinned runtime needs a compatible NVIDIA driver.")
        from numba_cuda_mlir import cuda
        if versions["numba-cuda-mlir"] != "0.6.0":
            raise RuntimeError("This course pins numba-cuda-mlir 0.6.0. Check the installation plan above.")
        if not cuda.is_available():
            raise RuntimeError("The selected Python kernel cannot access an NVIDIA GPU.")
        device = cuda.get_current_device()
        print(f"Native device: {device.name!s}; compute capability {device.compute_capability}")
        if tuple(device.compute_capability) < (7, 0):
            raise RuntimeError("The pinned backend requires compute capability 7.0 or later.")
    except Exception as error:
        print(f"Native execution unavailable: {type(error).__name__}: {error}")
        print("Check GPU allocation, driver and kernel environment. On Jetson, check JetPack and the ARM64 package plan.")
        return False
    print("Device detected. Run a program cell to verify compilation and execution on this device.")
    return True


def _task(case, key):
    task = LAB["tasks"][key]
    if task["case"] != case:
        raise ValueError(f"Task {key} belongs to case {task['case']}, not {case}.")
    return task


def _apply_edits(source, ids):
    for edit in ids:
        pairs = LAB["edits"][edit]
        for old, new in zip(pairs[::2], pairs[1::2]):
            source = source.replace(old, new)
    return source


def lab_start(case, key):
    """Return a fresh copy of the supplied source without modifying any cell."""
    starter = LAB["cases"][case]["starter"]
    if key == "exported":
        if case != LAB["exportedCase"]:
            raise ValueError("Choose the case named above Your exported program.")
        return LAB["exported"]
    if key == "starter":
        return starter
    return _apply_edits(starter, _task(case, key)["before"])


def _target(case, key):
    task = _task(case, key)
    if task["restore"]:
        return LAB["cases"][case]["starter"]
    return _apply_edits(lab_start(case, key), task["after"])


def _compare(actual, expected, spec):
    """Compare without converting the learner's output to the reference dtype or shape."""
    import numpy as np
    ref = np.array(expected, dtype=spec["dtype"]).reshape(spec["shape"])
    shape_ok = isinstance(actual, np.ndarray) and actual.shape == ref.shape
    dtype_ok = getattr(actual, "dtype", None) == ref.dtype
    soft = LAB["lesson"] == "softmax"
    rtol, atol = (1e-5, 1e-7) if soft else (0, 0)
    print(f"  Shape: {getattr(actual, 'shape', None)}; expected {ref.shape} · {'match' if shape_ok else 'different'}")
    print(f"  Dtype: {getattr(actual, 'dtype', None)}; expected {ref.dtype} · {'match' if dtype_ok else 'different'}")
    if not shape_ok or actual.dtype.kind not in "fiu":
        print("  Finiteness and values: unavailable for this shape or element type.")
        return False
    # Only authored overflow locations may be NaN. Infinity is never a reference value.
    finite_ok = np.array_equal(np.isnan(actual), np.isnan(ref)) and not np.isinf(actual).any()
    print(f"  Finiteness: {int(np.isfinite(actual).sum())}/{ref.size} finite; expected NaNs {int(np.isnan(ref).sum())} · {'match' if finite_ok else 'different'}")
    close = np.isclose(actual, ref, rtol=rtol, atol=atol, equal_nan=True) if soft else np.equal(actual, ref)
    print(f"  Values: {int(close.sum())}/{ref.size} match (rtol={rtol:g}, atol={atol:g}).")
    if not close.all():
        index = tuple(int(i) for i in np.argwhere(~close)[0])
        print(f"  First difference at {index}: observed {actual[index]}, expected {ref[index]}.")
    return bool(shape_ok and dtype_ok and finite_ok and close.all())


def _check_prints(lines, references):
    """The authored host prints follow the primary result, in source order."""
    import numpy as np
    if not references:
        return True
    extra = lines[1:]
    if len(extra) != len(references):
        print(f"  Additional output: expected {len(references)} lines, observed {len(extra)}.")
        return False
    for line, reference in zip(extra, references):
        if isinstance(reference, str):
            match = line.strip() == reference
        else:
            tokens = re.findall(r"[-+]?(?:nan|inf|(?:\d+(?:\.\d*)?|\.\d+)(?:e[-+]?\d+)?)", line.lower())
            values = np.array([float(t) for t in tokens])
            expected = np.asarray(reference, dtype=float)
            atol = 2e-6 if LAB["lesson"] == "softmax" else 0
            match = values.shape == expected.shape and np.allclose(values, expected, atol=atol, rtol=0, equal_nan=True)
        if not match:
            print(f"  Additional output differs: {line!r}; expected {reference}.")
            return False
    print("  Additional host output matches the requested observation.")
    return True


def _check_contract(actual, lines, spec, contract, baseline):
    import numpy as np
    policy = contract["policy"]
    if policy == "race":
        valid = isinstance(actual, np.ndarray) and actual.dtype == np.dtype(spec["dtype"]) and list(actual.shape) == spec["shape"]
        if not valid:
            print("The observed counters have the wrong shape or dtype.")
            return "fail"
        hot = int(np.argmax(spec["expected"]))
        if np.any(np.delete(actual, hot) != 0) or not 1 <= actual[hot] <= spec["expected"][hot]:
            print("Counters fall outside this hot-bin experiment: only the routed bin receives increments.")
            return "fail"
        if not _check_prints(lines, contract["prints"]):
            return "fail"
        print(f"Observed counters: {actual.tolist()}. Repeat and record bin 2. Even an exact starter match cannot establish atomicity or race safety.")
        return "observation"
    ok = baseline if contract["values"] == spec["expected"] else _compare(actual, contract["values"], spec)
    if policy == "probabilities" and isinstance(actual, np.ndarray) and list(actual.shape) == spec["shape"] and actual.dtype.kind == "f":
        sums = actual.sum(axis=1, dtype=np.float64)
        print("  Probability row sums:", sums, "(absolute tolerance 2e-6); each probability must be in [0, 1].")
        ok = ok and bool(((actual >= 0) & (actual <= 1)).all() and np.allclose(sums, 1, atol=2e-6, rtol=0))
    if policy == "overflow":
        print("  The unshifted exponential overflows or underflows on these inputs. Compare the NaN locations above.")
    if policy == "precision" and isinstance(actual, np.ndarray) and list(actual.shape) == spec["shape"] and actual.dtype.kind in "fiu":
        exact = np.array(contract["values"])
        print("  Absolute error versus the mathematical sum: starter", abs(np.array(spec["expected"]) - exact), "this run", abs(actual - exact))
    prints_ok = _check_prints(lines, contract["prints"])
    return "pass" if ok and prints_ok else "fail"


def assess(case, key, source, actual, lines):
    spec = LAB["cases"][case]
    # IPython may supply one fewer terminal newline to a cell magic.
    body = source.rstrip("\n")
    starter = spec["starter"].rstrip("\n")
    print("Starter comparison")
    baseline = _compare(actual, spec["expected"], spec)
    task_key = key
    if key == "exported":
        selected = LAB["selectedTask"]
        candidates = [k for k, t in LAB["tasks"].items() if t["case"] == case and t["kind"] == "run"]
        if selected:
            task_key = selected
        elif body == starter:
            task_key = "starter"
        else:
            task_key = next((k for k in candidates if body == _target(case, k).rstrip("\n")), "starter")
    starting = False
    if task_key == "starter":
        if body != starter:
            print("Objective unassessed: this custom program has no matching authored reference.")
            return "unassessed"
        contract = {"policy":"probabilities" if LAB["lesson"] == "softmax" else "values", "values":spec["expected"], "prints":[]}
    else:
        t = _task(case, task_key)
        if t["kind"] == "reason":
            print("Source reasoning: explain the task using the supplied code.")
            return "reasoning"
        if body == _target(case, task_key).rstrip("\n"):
            contract = t["objective"]
        elif body == lab_start(case, task_key).rstrip("\n"):
            starting = True
            contract = t["startingObjective"]
        else:
            print("Objective unassessed: the program differs from the authored edit for this task.")
            return "unassessed"

    print(f"Starting result · {task_key}" if starting else f"Objective · {task_key}")
    verdict = _check_contract(actual, lines, spec, contract, baseline)
    if starting:
        if verdict == "fail":
            print("Starting result failed its check. Inspect the differences above before making the task edit.")
            return "starting-fail"
        if verdict == "observation":
            print("Starting observation recorded within the experiment bounds. Make the requested edit and rerun.")
            return "starting-observation"
        print("Starting result verified. Make the requested edit in this cell, then rerun to check its objective.")
        return "starting-point"
    if verdict != "observation":
        print(f"Numerical objective: {verdict}. Explain the source or measurements requested in the task.")
    return verdict


def execute_program(source):
    """IPython registry-independent execution, with source available for JIT inspection."""
    filename = "<cuda-lab-" + hashlib.sha256(source.encode()).hexdigest() + ">"
    linecache.cache[filename] = (len(source), None, source.splitlines(keepends=True), filename)
    scope = {"__name__": "__main__", "__file__": filename}
    output = io.StringIO()
    try:
        with contextlib.redirect_stdout(output):
            exec(compile(source, filename, "exec"), scope)
    finally:
        print(output.getvalue(), end="")
    return scope.get("actual"), output.getvalue().splitlines()


def run_experiment(line, cell):
    lesson, case, key = line.split()
    if lesson != LAB["lesson"]:
        raise ValueError("Run the setup cell from this notebook before its program cells.")
    lab_start(case, key)  # Validate the fixture/task binding before executing user code.
    print(f"{lesson} / {case} / {key} · source SHA-256 prefix {hashlib.sha256(cell.encode()).hexdigest()[:16]}")
    if not NATIVE_READY:
        raise RuntimeError("Native preflight did not pass. Review setup and rerun it. No result was checked.")
    try:
        actual, lines = execute_program(cell)
    except Exception:
        print("Program execution failed. No result was checked.")
        raise
    if actual is None:
        print("This run did not assign a returned array to actual.")
    assess(case, key, cell, actual, lines)

LAB = json.loads("{\"lesson\":\"softmax\",\"exportedCase\":\"wide-37\",\"exported\":\"import math\\n\\nfrom numba_cuda_mlir import cuda\\nfrom numba_cuda_mlir.types import float32\\n\\n\\n@cuda.jit\\ndef kernel(values, out, rows, cols):\\n    scratch = cuda.shared.array(32, dtype=float32)\\n    row, lane = cuda.blockIdx.x, cuda.threadIdx.x\\n    if row >= rows:\\n        return\\n\\n    local_max = -math.inf\\n    for col in range(lane, cols, cuda.blockDim.x):\\n        local_max = max(local_max, values[row, col])\\n    scratch[lane] = local_max\\n    cuda.syncthreads()\\n\\n    distance = cuda.blockDim.x // 2\\n    while distance > 0:\\n        if lane < distance:\\n            scratch[lane] = max(scratch[lane], scratch[lane + distance])\\n        cuda.syncthreads()\\n        distance //= 2\\n\\n    row_max = scratch[0]\\n    cuda.syncthreads()\\n    local_sum = float32(0)\\n    for col in range(lane, cols, cuda.blockDim.x):\\n        local_sum += math.exp(values[row, col] - row_max)\\n    scratch[lane] = local_sum\\n    cuda.syncthreads()\\n\\n    distance = cuda.blockDim.x // 2\\n    while distance > 0:\\n        if lane < distance:\\n            scratch[lane] += scratch[lane + distance]\\n        cuda.syncthreads()\\n        distance //= 2\\n\\n    for col in range(lane, cols, cuda.blockDim.x):\\n        out[row, col] = math.exp(values[row, col] - row_max) / scratch[0]\\n\\n\\n# Host code\\ndef main():\\n    import numpy as np\\n    from numba_cuda_mlir import cuda\\n\\n    values_host = np.array([\\n        -4, -3, -2, -1, 0, 1, 2, 3,\\n        4, -4, -3, -2, -1, 0, 1, 2,\\n        3, 4, -4, -3, -2, -1, 0, 1,\\n        2, 3, 4, -4, -3, -2, -1, 0,\\n        1, 2, 3, 4, -4, 800, 801, 802,\\n        803, 804, 800, 801, 802, 803, 804, 800,\\n        801, 802, 803, 804, 800, 801, 802, 803,\\n        804, 800, 801, 802, 803, 804, 800, 801,\\n        802, 803, 804, 800, 801, 802, 803, 804,\\n        800, 801,\\n    ], dtype=np.float32).reshape(2, 37)\\n    values = cuda.to_device(values_host)\\n\\n    out_host = np.zeros((2, 37), dtype=np.float32)\\n    out = cuda.to_device(out_host)\\n\\n    kernel[2, 32](values, out, 2, 37)\\n    result = out.copy_to_host()\\n    print(\\\"out =\\\", result.tolist())\\n    return result\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    actual = main()\\n\",\"selectedTask\":\"lane\",\"cases\":{\"extreme\":{\"label\":\"Extreme and equal logits\",\"shape\":[2,5],\"dtype\":\"float32\",\"expected\":[0.4136219764199625,0.1521630215416033,0.4136219764199625,0.02059302561847168,0,0.053671883236681636,0.14589530490144734,0.3965845561710961,0.3965845561710961,0.007263699519678713],\"starter\":\"import math\\n\\nfrom numba_cuda_mlir import cuda\\nfrom numba_cuda_mlir.types import float32\\n\\n\\n@cuda.jit\\ndef kernel(values, out, rows, cols):\\n    scratch = cuda.shared.array(32, dtype=float32)\\n    row, lane = cuda.blockIdx.x, cuda.threadIdx.x\\n    if row >= rows:\\n        return\\n\\n    local_max = -math.inf\\n    for col in range(lane, cols, cuda.blockDim.x):\\n        local_max = max(local_max, values[row, col])\\n    scratch[lane] = local_max\\n    cuda.syncthreads()\\n\\n    distance = cuda.blockDim.x // 2\\n    while distance > 0:\\n        if lane < distance:\\n            scratch[lane] = max(scratch[lane], scratch[lane + distance])\\n        cuda.syncthreads()\\n        distance //= 2\\n\\n    row_max = scratch[0]\\n    cuda.syncthreads()\\n    local_sum = float32(0)\\n    for col in range(lane, cols, cuda.blockDim.x):\\n        local_sum += math.exp(values[row, col] - row_max)\\n    scratch[lane] = local_sum\\n    cuda.syncthreads()\\n\\n    distance = cuda.blockDim.x // 2\\n    while distance > 0:\\n        if lane < distance:\\n            scratch[lane] += scratch[lane + distance]\\n        cuda.syncthreads()\\n        distance //= 2\\n\\n    for col in range(lane, cols, cuda.blockDim.x):\\n        out[row, col] = math.exp(values[row, col] - row_max) / scratch[0]\\n\\n\\n# Host code\\ndef main():\\n    import numpy as np\\n    from numba_cuda_mlir import cuda\\n\\n    values_host = np.array([\\n        1000, 999, 1000, 997, -1000, -1002, -1001, -1000,\\n        -1000, -1004,\\n    ], dtype=np.float32).reshape(2, 5)\\n    values = cuda.to_device(values_host)\\n\\n    out_host = np.zeros((2, 5), dtype=np.float32)\\n    out = cuda.to_device(out_host)\\n\\n    kernel[2, 32](values, out, 2, 5)\\n    result = out.copy_to_host()\\n    print(\\\"out =\\\", result.tolist())\\n    return result\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    actual = main()\\n\"},\"wide-37\":{\"label\":\"Two rows · 37 columns\",\"shape\":[2,37],\"dtype\":\"float32\",\"expected\":[0.000053016938166048376,0.00014411497961730612,0.0003917451303024689,0.001064873669088522,0.0028946267442878397,0.007868411279169204,0.02138855939900784,0.05814013235123992,0.15804126527457935,0.000053016938166048376,0.00014411497961730612,0.0003917451303024689,0.001064873669088522,0.0028946267442878397,0.007868411279169204,0.02138855939900784,0.05814013235123992,0.15804126527457935,0.000053016938166048376,0.00014411497961730612,0.0003917451303024689,0.001064873669088522,0.0028946267442878397,0.007868411279169204,0.02138855939900784,0.05814013235123992,0.15804126527457935,0.000053016938166048376,0.00014411497961730612,0.0003917451303024689,0.001064873669088522,0.0028946267442878397,0.007868411279169204,0.02138855939900784,0.05814013235123992,0.15804126527457935,0.000053016938166048376,0.001654929202618547,0.004498563978864214,0.012228364717907015,0.033240141604456354,0.09035607289879921,0.001654929202618547,0.004498563978864214,0.012228364717907015,0.033240141604456354,0.09035607289879921,0.001654929202618547,0.004498563978864214,0.012228364717907015,0.033240141604456354,0.09035607289879921,0.001654929202618547,0.004498563978864214,0.012228364717907015,0.033240141604456354,0.09035607289879921,0.001654929202618547,0.004498563978864214,0.012228364717907015,0.033240141604456354,0.09035607289879921,0.001654929202618547,0.004498563978864214,0.012228364717907015,0.033240141604456354,0.09035607289879921,0.001654929202618547,0.004498563978864214,0.012228364717907015,0.033240141604456354,0.09035607289879921,0.001654929202618547,0.004498563978864214],\"starter\":\"import math\\n\\nfrom numba_cuda_mlir import cuda\\nfrom numba_cuda_mlir.types import float32\\n\\n\\n@cuda.jit\\ndef kernel(values, out, rows, cols):\\n    scratch = cuda.shared.array(32, dtype=float32)\\n    row, lane = cuda.blockIdx.x, cuda.threadIdx.x\\n    if row >= rows:\\n        return\\n\\n    local_max = -math.inf\\n    for col in range(lane, cols, cuda.blockDim.x):\\n        local_max = max(local_max, values[row, col])\\n    scratch[lane] = local_max\\n    cuda.syncthreads()\\n\\n    distance = cuda.blockDim.x // 2\\n    while distance > 0:\\n        if lane < distance:\\n            scratch[lane] = max(scratch[lane], scratch[lane + distance])\\n        cuda.syncthreads()\\n        distance //= 2\\n\\n    row_max = scratch[0]\\n    cuda.syncthreads()\\n    local_sum = float32(0)\\n    for col in range(lane, cols, cuda.blockDim.x):\\n        local_sum += math.exp(values[row, col] - row_max)\\n    scratch[lane] = local_sum\\n    cuda.syncthreads()\\n\\n    distance = cuda.blockDim.x // 2\\n    while distance > 0:\\n        if lane < distance:\\n            scratch[lane] += scratch[lane + distance]\\n        cuda.syncthreads()\\n        distance //= 2\\n\\n    for col in range(lane, cols, cuda.blockDim.x):\\n        out[row, col] = math.exp(values[row, col] - row_max) / scratch[0]\\n\\n\\n# Host code\\ndef main():\\n    import numpy as np\\n    from numba_cuda_mlir import cuda\\n\\n    values_host = np.array([\\n        -4, -3, -2, -1, 0, 1, 2, 3,\\n        4, -4, -3, -2, -1, 0, 1, 2,\\n        3, 4, -4, -3, -2, -1, 0, 1,\\n        2, 3, 4, -4, -3, -2, -1, 0,\\n        1, 2, 3, 4, -4, 800, 801, 802,\\n        803, 804, 800, 801, 802, 803, 804, 800,\\n        801, 802, 803, 804, 800, 801, 802, 803,\\n        804, 800, 801, 802, 803, 804, 800, 801,\\n        802, 803, 804, 800, 801, 802, 803, 804,\\n        800, 801,\\n    ], dtype=np.float32).reshape(2, 37)\\n    values = cuda.to_device(values_host)\\n\\n    out_host = np.zeros((2, 37), dtype=np.float32)\\n    out = cuda.to_device(out_host)\\n\\n    kernel[2, 32](values, out, 2, 37)\\n    result = out.copy_to_host()\\n    print(\\\"out =\\\", result.tolist())\\n    return result\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    actual = main()\\n\"}},\"tasks\":{\"sums\":{\"case\":\"extreme\",\"kind\":\"run\",\"before\":[],\"after\":[\"sumRows\"],\"restore\":false,\"objective\":{\"values\":[0.4136219764199625,0.1521630215416033,0.4136219764199625,0.02059302561847168,0,0.053671883236681636,0.14589530490144734,0.3965845561710961,0.3965845561710961,0.007263699519678713],\"policy\":\"probabilities\",\"prints\":[[1,1]]},\"startingObjective\":{\"values\":[0.4136219764199625,0.1521630215416033,0.4136219764199625,0.02059302561847168,0,0.053671883236681636,0.14589530490144734,0.3965845561710961,0.3965845561710961,0.007263699519678713],\"policy\":\"probabilities\",\"prints\":[]}},\"overflow\":{\"case\":\"extreme\",\"kind\":\"run\",\"before\":[\"sumRows\"],\"after\":[\"unshift\",\"finite\"],\"restore\":false,\"objective\":{\"values\":[null,null,null,null,0,null,null,null,null,null],\"policy\":\"overflow\",\"prints\":[[null,null],\"False\"]},\"startingObjective\":{\"values\":[0.4136219764199625,0.1521630215416033,0.4136219764199625,0.02059302561847168,0,0.053671883236681636,0.14589530490144734,0.3965845561710961,0.3965845561710961,0.007263699519678713],\"policy\":\"probabilities\",\"prints\":[[1,1]]}},\"lane\":{\"case\":\"wide-37\",\"kind\":\"run\",\"before\":[],\"after\":[],\"restore\":false,\"objective\":{\"values\":[0.000053016938166048376,0.00014411497961730612,0.0003917451303024689,0.001064873669088522,0.0028946267442878397,0.007868411279169204,0.02138855939900784,0.05814013235123992,0.15804126527457935,0.000053016938166048376,0.00014411497961730612,0.0003917451303024689,0.001064873669088522,0.0028946267442878397,0.007868411279169204,0.02138855939900784,0.05814013235123992,0.15804126527457935,0.000053016938166048376,0.00014411497961730612,0.0003917451303024689,0.001064873669088522,0.0028946267442878397,0.007868411279169204,0.02138855939900784,0.05814013235123992,0.15804126527457935,0.000053016938166048376,0.00014411497961730612,0.0003917451303024689,0.001064873669088522,0.0028946267442878397,0.007868411279169204,0.02138855939900784,0.05814013235123992,0.15804126527457935,0.000053016938166048376,0.001654929202618547,0.004498563978864214,0.012228364717907015,0.033240141604456354,0.09035607289879921,0.001654929202618547,0.004498563978864214,0.012228364717907015,0.033240141604456354,0.09035607289879921,0.001654929202618547,0.004498563978864214,0.012228364717907015,0.033240141604456354,0.09035607289879921,0.001654929202618547,0.004498563978864214,0.012228364717907015,0.033240141604456354,0.09035607289879921,0.001654929202618547,0.004498563978864214,0.012228364717907015,0.033240141604456354,0.09035607289879921,0.001654929202618547,0.004498563978864214,0.012228364717907015,0.033240141604456354,0.09035607289879921,0.001654929202618547,0.004498563978864214,0.012228364717907015,0.033240141604456354,0.09035607289879921,0.001654929202618547,0.004498563978864214],\"policy\":\"probabilities\",\"prints\":[]}}},\"edits\":{\"sumRows\":[\"    return result\",\"    print(result.sum(axis=1))\\n    return result\"],\"unshift\":[\"math.exp(values[row, col] - row_max)\",\"math.exp(values[row, col])\"],\"finite\":[\"    return result\",\"    print(np.isfinite(result).all())\\n    return result\"]}}")
get_ipython().register_magic_function(run_experiment, "cell", "experiment")
NATIVE_READY = lab_preflight()


## Your exported program

**Two rows · 37 columns** · `wide-37`

Selected task at export: **Trace the stable 37-column row**. This copy contains the open editor text, including host changes. Recognized legacy CUDA imports are adapted for the native backend. The numbered experiments and starter cases use course source. Other saved case edits stay in your browser. Use **Editor tools → Export all learner work** there for a full backup.

The checker compares a recognized program with its case reference. For custom code it shows a starter comparison and leaves the custom objective unassessed. Shape and dtype must match exactly. Float32 exponentials and parallel sums can round slightly differently. Each probability has absolute tolerance 1e-7 plus relative tolerance 1e-5 times the reference magnitude. Stable row sums must be within 2e-6 of one. To supply a result, assign the returned NumPy array to `actual` in the program.

In [ ]:
%%experiment softmax wide-37 exported
import math

from numba_cuda_mlir import cuda
from numba_cuda_mlir.types import float32


@cuda.jit
def kernel(values, out, rows, cols):
    scratch = cuda.shared.array(32, dtype=float32)
    row, lane = cuda.blockIdx.x, cuda.threadIdx.x
    if row >= rows:
        return

    local_max = -math.inf
    for col in range(lane, cols, cuda.blockDim.x):
        local_max = max(local_max, values[row, col])
    scratch[lane] = local_max
    cuda.syncthreads()

    distance = cuda.blockDim.x // 2
    while distance > 0:
        if lane < distance:
            scratch[lane] = max(scratch[lane], scratch[lane + distance])
        cuda.syncthreads()
        distance //= 2

    row_max = scratch[0]
    cuda.syncthreads()
    local_sum = float32(0)
    for col in range(lane, cols, cuda.blockDim.x):
        local_sum += math.exp(values[row, col] - row_max)
    scratch[lane] = local_sum
    cuda.syncthreads()

    distance = cuda.blockDim.x // 2
    while distance > 0:
        if lane < distance:
            scratch[lane] += scratch[lane + distance]
        cuda.syncthreads()
        distance //= 2

    for col in range(lane, cols, cuda.blockDim.x):
        out[row, col] = math.exp(values[row, col] - row_max) / scratch[0]


# Host code
def main():
    import numpy as np
    from numba_cuda_mlir import cuda

    values_host = np.array([
        -4, -3, -2, -1, 0, 1, 2, 3,
        4, -4, -3, -2, -1, 0, 1, 2,
        3, 4, -4, -3, -2, -1, 0, 1,
        2, 3, 4, -4, -3, -2, -1, 0,
        1, 2, 3, 4, -4, 800, 801, 802,
        803, 804, 800, 801, 802, 803, 804, 800,
        801, 802, 803, 804, 800, 801, 802, 803,
        804, 800, 801, 802, 803, 804, 800, 801,
        802, 803, 804, 800, 801, 802, 803, 804,
        800, 801,
    ], dtype=np.float32).reshape(2, 37)
    values = cuda.to_device(values_host)

    out_host = np.zeros((2, 37), dtype=np.float32)
    out = cuda.to_device(out_host)

    kernel[2, 32](values, out, 2, 37)
    result = out.copy_to_host()
    print("out =", result.tolist())
    return result


if __name__ == "__main__":
    actual = main()


## 1. Check row normalization

**Starter case** · Extreme and equal logits

**Predict for the starter case.** Row 0 is [1000, 999, 1000, 997, −1000]. What is the exponential of the largest score after subtracting the row maximum? Which two columns should have equal probabilities?

Start with the supplied case program. Run Extreme and equal logits. Compare the first and third probabilities in row 0. In main(), add print(result.sum(axis=1)) just before return result and check how close both row sums are to 1.

**Evidence to examine.** Compare columns 0 and 2 and read both printed sums.

In [ ]:
%%experiment softmax extreme sums
import math

from numba_cuda_mlir import cuda
from numba_cuda_mlir.types import float32


@cuda.jit
def kernel(values, out, rows, cols):
    scratch = cuda.shared.array(32, dtype=float32)
    row, lane = cuda.blockIdx.x, cuda.threadIdx.x
    if row >= rows:
        return

    local_max = -math.inf
    for col in range(lane, cols, cuda.blockDim.x):
        local_max = max(local_max, values[row, col])
    scratch[lane] = local_max
    cuda.syncthreads()

    distance = cuda.blockDim.x // 2
    while distance > 0:
        if lane < distance:
            scratch[lane] = max(scratch[lane], scratch[lane + distance])
        cuda.syncthreads()
        distance //= 2

    row_max = scratch[0]
    cuda.syncthreads()
    local_sum = float32(0)
    for col in range(lane, cols, cuda.blockDim.x):
        local_sum += math.exp(values[row, col] - row_max)
    scratch[lane] = local_sum
    cuda.syncthreads()

    distance = cuda.blockDim.x // 2
    while distance > 0:
        if lane < distance:
            scratch[lane] += scratch[lane + distance]
        cuda.syncthreads()
        distance //= 2

    for col in range(lane, cols, cuda.blockDim.x):
        out[row, col] = math.exp(values[row, col] - row_max) / scratch[0]


# Host code
def main():
    import numpy as np
    from numba_cuda_mlir import cuda

    values_host = np.array([
        1000, 999, 1000, 997, -1000, -1002, -1001, -1000,
        -1000, -1004,
    ], dtype=np.float32).reshape(2, 5)
    values = cuda.to_device(values_host)

    out_host = np.zeros((2, 5), dtype=np.float32)
    out = cuda.to_device(out_host)

    kernel[2, 32](values, out, 2, 5)
    result = out.copy_to_host()
    print("out =", result.tolist())
    return result


if __name__ == "__main__":
    actual = main()


## 2. Remove the maximum shift

**Starter case** · Extreme and equal logits

**Predict for the starter case.** Row 0 is [1000, 999, 1000, 997, −1000]. What is the exponential of the largest score after subtracting the row maximum? Which two columns should have equal probabilities?

This copy includes the prerequisite edit. Replace both math.exp(values[row, col] - row_max) calls with math.exp(values[row, col]). In main(), add print(np.isfinite(result).all()) just before return result and rerun. Does the result still contain only finite values?

**Evidence to examine.** Check the finiteness flag. Nonfinite probabilities are evidence for this edit.

In [ ]:
%%experiment softmax extreme overflow
import math

from numba_cuda_mlir import cuda
from numba_cuda_mlir.types import float32


@cuda.jit
def kernel(values, out, rows, cols):
    scratch = cuda.shared.array(32, dtype=float32)
    row, lane = cuda.blockIdx.x, cuda.threadIdx.x
    if row >= rows:
        return

    local_max = -math.inf
    for col in range(lane, cols, cuda.blockDim.x):
        local_max = max(local_max, values[row, col])
    scratch[lane] = local_max
    cuda.syncthreads()

    distance = cuda.blockDim.x // 2
    while distance > 0:
        if lane < distance:
            scratch[lane] = max(scratch[lane], scratch[lane + distance])
        cuda.syncthreads()
        distance //= 2

    row_max = scratch[0]
    cuda.syncthreads()
    local_sum = float32(0)
    for col in range(lane, cols, cuda.blockDim.x):
        local_sum += math.exp(values[row, col] - row_max)
    scratch[lane] = local_sum
    cuda.syncthreads()

    distance = cuda.blockDim.x // 2
    while distance > 0:
        if lane < distance:
            scratch[lane] += scratch[lane + distance]
        cuda.syncthreads()
        distance //= 2

    for col in range(lane, cols, cuda.blockDim.x):
        out[row, col] = math.exp(values[row, col] - row_max) / scratch[0]


# Host code
def main():
    import numpy as np
    from numba_cuda_mlir import cuda

    values_host = np.array([
        1000, 999, 1000, 997, -1000, -1002, -1001, -1000,
        -1000, -1004,
    ], dtype=np.float32).reshape(2, 5)
    values = cuda.to_device(values_host)

    out_host = np.zeros((2, 5), dtype=np.float32)
    out = cuda.to_device(out_host)

    kernel[2, 32](values, out, 2, 5)
    result = out.copy_to_host()
    print("out =", result.tolist())
    print(result.sum(axis=1))
    return result


if __name__ == "__main__":
    actual = main()


## 3. Trace the stable 37-column row

**Starter case** · Two rows · 37 columns

**Predict for the starter case.** With 37 columns and 32 threads per row, which columns does Lane 0 write? What should each stable probability row sum to?

Start with the supplied case program. Run Two rows · 37 columns with the starter. In the final loop, list the columns where Lane 0 writes. How does the number of threads per block set their spacing?

**Evidence to examine.** Read the finite row output and derive Lane 0’s columns from the final loop.

In [ ]:
%%experiment softmax wide-37 lane
import math

from numba_cuda_mlir import cuda
from numba_cuda_mlir.types import float32


@cuda.jit
def kernel(values, out, rows, cols):
    scratch = cuda.shared.array(32, dtype=float32)
    row, lane = cuda.blockIdx.x, cuda.threadIdx.x
    if row >= rows:
        return

    local_max = -math.inf
    for col in range(lane, cols, cuda.blockDim.x):
        local_max = max(local_max, values[row, col])
    scratch[lane] = local_max
    cuda.syncthreads()

    distance = cuda.blockDim.x // 2
    while distance > 0:
        if lane < distance:
            scratch[lane] = max(scratch[lane], scratch[lane + distance])
        cuda.syncthreads()
        distance //= 2

    row_max = scratch[0]
    cuda.syncthreads()
    local_sum = float32(0)
    for col in range(lane, cols, cuda.blockDim.x):
        local_sum += math.exp(values[row, col] - row_max)
    scratch[lane] = local_sum
    cuda.syncthreads()

    distance = cuda.blockDim.x // 2
    while distance > 0:
        if lane < distance:
            scratch[lane] += scratch[lane + distance]
        cuda.syncthreads()
        distance //= 2

    for col in range(lane, cols, cuda.blockDim.x):
        out[row, col] = math.exp(values[row, col] - row_max) / scratch[0]


# Host code
def main():
    import numpy as np
    from numba_cuda_mlir import cuda

    values_host = np.array([
        -4, -3, -2, -1, 0, 1, 2, 3,
        4, -4, -3, -2, -1, 0, 1, 2,
        3, 4, -4, -3, -2, -1, 0, 1,
        2, 3, 4, -4, -3, -2, -1, 0,
        1, 2, 3, 4, -4, 800, 801, 802,
        803, 804, 800, 801, 802, 803, 804, 800,
        801, 802, 803, 804, 800, 801, 802, 803,
        804, 800, 801, 802, 803, 804, 800, 801,
        802, 803, 804, 800, 801, 802, 803, 804,
        800, 801,
    ], dtype=np.float32).reshape(2, 37)
    values = cuda.to_device(values_host)

    out_host = np.zeros((2, 37), dtype=np.float32)
    out = cuda.to_device(out_host)

    kernel[2, 32](values, out, 2, 37)
    result = out.copy_to_host()
    print("out =", result.tolist())
    return result


if __name__ == "__main__":
    actual = main()


## Explain the authored examples

Removing the shift makes exp(1000) overflow. Dividing an infinite exponential by an infinite total can produce NaN. Columns 0 and 2 get equal probabilities in the stable version because they start at the same score and share the divisor. With 37 columns, Lane 0 visits columns 0 and 32.

### Optional native investigation

In Appendix A, run the softmax investigation against torch.softmax on the same float32 buffers. Check the probabilities before comparing CUDA event times.

The optional Appendix A investigation uses the [native investigation files](https://snowch.github.io/cuda-python-lab/gpu/cuda-python-investigations.zip) and the tools named in their instructions.